###Silver data transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta")\
        .option("header", True)\
        .option("inferSchema", True)\
        .load("abfss://bronze@pravdatalake.dfs.core.windows.net/netflix_titles")

In [0]:
df.printSchema()

In [0]:

df = df.withColumn(
    "duration_minutes",
    expr("try_cast(duration_minutes AS INT)")
).withColumn(
    "duration_seasons",
    expr("try_cast(duration_seasons AS INT)")
)

In [0]:
df.display()

In [0]:
df = df.fillna({"duration_minutes" : 0, "duration_seasons" : 1})

In [0]:
df.display()

In [0]:
df = df.withColumn("small_title", split(col("title"),":")[0])


In [0]:
df = df.withColumn("rating",split(col("rating"),"-")[0])


In [0]:
df = df.withColumn("type_flag", when(col('type')=='Movie',1)\
    .when(col('type')=='TV Show',2)\
    .otherwise(0) )
display(df)

In [0]:
from pyspark.sql.window import Window

In [0]:
df = df.withColumn("duration_ranking", dense_rank().over(Window.orderBy(col("duration_minutes").desc())))

In [0]:
df.display()

In [0]:
df.createOrReplaceTempView("temp_view")

In [0]:
df = spark.sql("""
               SELECT * FROM temp_view
               """)

In [0]:
df.display()

In [0]:
df_visual = df.groupBy("type").agg(count("*").alias("total_count"))


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("path", "abfss://silver@pravdatalake.dfs.core.windows.net/netflix_titles")\
       .save() 